# Testing grounds

In [1]:
# Necessary imports
import matplotlib.pyplot as plt
from src.data.dataset_manager_factory import DatasetManagerFactory
from src.models.model_configuration import ModelConfiguration
from src.logs.logs_manager import LogsManager
from src.models.models_manager import ModelsManager
from src.attacks.attacks_manager import AttacksManager
from src.models.lenet import LeNet
import torch
import numpy as np

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

## Dataset

In [ ]:
# Load the dataset using DataManager
data_manager = DatasetManagerFactory('MNIST')
# train_split, val_split = data_manager.calculate_split_sizes(1, 0)
train_dataset, val_dataset = data_manager.split_dataset(
    train_samples=500, val_samples=150, batch_size=64)

## Models Training

In [2]:
# Create services instance
logs_manager = LogsManager()
model_manager = ModelsManager(configuration)
attack_manager = AttacksManager()

# Register models
model1 = LeNet('LeNet_Test_1', numChannels=1, classes=10)
# Train the model
logging.info(f"Training model '{model_name}'...")
trainer.fit(model_instance, train_dataset, val_dataset)
training_results[model_name] = trainer.logged_metrics

# Test the model
if test_dataset is None: continue
logging.info(f"Testing model '{model_name}'...")
test_results = trainer.test(model_instance, dataloaders=test_dataset)[0]
evaluation_results[model_name] = test_results


model2 = LeNet('LeNet_Test_2', numChannels=1, classes=10)

## Attacks Generation

In [3]:
# Example attack generation code
# Assuming attack_manager has a method to generate attacks
x_test, y_test = val_dataset
attacker_pgd = attack_manager.create_attack('pgd', model1)
attacker_robust_pgd = attack_manager.create_attack('pgd', model2)

eps_range = [0.01, 0.02, 0.03, 0.04, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6]
nb_correct_original = []
nb_correct_robust = []

nb_samples = 100

for eps in eps_range:
    attacker_pgd.set_params(**{'eps': eps})
    attacker_robust_pgd.set_params(**{'eps': eps})
    x_test_adv = attacker_pgd.generate(x_test[:nb_samples], y_test[:nb_samples])
    x_test_adv_robust = attacker_robust_pgd.generate(x_test[:nb_samples], y_test[:nb_samples])
    
    x_test_adv_pred = np.argmax(model1.predict(x_test_adv), axis=1)
    nb_correct_original += [np.sum(x_test_adv_pred == np.argmax(y_test[:nb_samples], axis=1)) / nb_samples]
    
    x_test_adv_robust_pred = np.argmax(model2.predict(x_test_adv_robust), axis=1)
    nb_correct_robust += [np.sum(x_test_adv_robust_pred == np.argmax(y_test[:nb_samples], axis=1)) / nb_samples]


## Logs Generation

In [4]:
# Example logs generation code
fig, ax = plt.subplots()
ax.plot(np.array(eps_range), np.array(nb_correct_original), 'b--', label='Original classifier')
ax.plot(np.array(eps_range), np.array(nb_correct_robust), 'r--', label='Robust classifier')

legend = ax.legend(loc='upper right', shadow=True, fontsize='large')

plt.xlabel('Perturbation size (eps, L-Inf)')
plt.ylabel('Classification Accuracy')
plt.show()

# Save the results to the ./results directory
plt.savefig(f"./results/adversarial_examples.png")
